# 01 — Generate & Visualize a Synthetic Bio-Topology

This notebook takes you from zero to an interactive D3 topology graph.
No data download, no server setup — just Python 3.10+ and numpy.

**Prerequisites:**
- Python 3.10+ with numpy installed
- Running from inside the `flybrain-connectome` checkout

```bash
pip install numpy
```

## Cell 1 — Import and configure

In [ ]:
import sys, os
from pathlib import Path

# Make the toolkit importable
HERE = Path(__file__).resolve().parents[1]  # repo root
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import synthetic_bio_topology as c1
print("[ok] toolkit loaded from:", c1.__file__)

## Cell 2 — Design a topology

In [ ]:
spec = {
    "n_modules": 5,
    "agents_per_module": 8,
}

topo = c1.design_organism(spec)
print("Agents:", len(topo["agents"]))
print("Edges :", len(topo["edges"]))
print("Modules:", len(topo["modules"]))

## Cell 3 — Inspect the structure

In [ ]:
# Look at the first 3 agents
for a in topo["agents"][:3]:
    print(f"  {a['id']}  role={a.get('role','?')}  module={a.get('module','?')}")

# Look at the first 5 edges
print()
for e in topo["edges"][:5]:
    src = e["source"]
    tgt = e["target"]
    prov = e.get("provenance", {})
    src_neuron = prov.get("source_neuron", "?")
    print(f"  {src} -> {tgt}  via {src_neuron}")

## Cell 4 — Save and render HTML

In [ ]:
import json

name = "notebook_demo_m5_a8"
org_dir = TOOLKIT / "generated_organisms"
org_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
json_path = org_dir / (name + ".json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(topo, f, indent=2, ensure_ascii=False, default=str)
print("[saved]", json_path)

# Render HTML
html_path = org_dir / (name + ".html")
try:
    html = c1.render_html(topo)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    print("[saved]", html_path)
except (ImportError, AttributeError) as e:
    print("[skip] HTML render not available:", e)

## Cell 5 — Open the visualization

In [ ]:
import webbrowser
webbrowser.open(str(html_path))
print("[ok] Opened in browser")

## What you should see

A **force-directed D3 graph** with:
- 5 color-coded modules
- 8 agents per module (40 total)
- Inhibitory coordinator (red hub) connecting modules
- Reservoir buffer (blue) for cross-module memory
- **Draggable nodes** and **hover tooltips**
- **Dark theme** with no CDN dependencies (fully offline)

Every edge has a `provenance` field tracing it to a real *Drosophila*
neuron or synapse class — the core value proposition of this project.

## Next

- [02_export_swarmlabs.ipynb](02_export_swarmlabs.ipynb) — export to SwarmLabs / CrewAI config
- [03_provenance_audit.ipynb](03_provenance_audit.ipynb) — query the provenance chain